# 🧭 Chapter 06-02a: Tool Calling & Function Calling Deep Dive

---

## 📖 The Story: Teaching Robots to Use Gadgets

An LLM with tools is like a chef with a kitchen. Without tools, they can only describe recipes. With tools — oven, blender, thermometer — they can actually cook.

Function calling is the protocol that lets LLMs say: *"I need to use the thermometer now, set to Celsius."*

---

## 🎯 Function Calling: The Protocol

```
╔══════════════════════════════════════════════════════════════════════╗
║  FUNCTION CALLING                                                   ║
║                                                                      ║
║  A protocol where the LLM outputs structured tool invocations      ║
║  instead of (or alongside) text. The APPLICATION executes the      ║
║  function and returns results to the LLM.                          ║
║                                                                      ║
║  Key: The LLM NEVER executes functions — it only REQUESTS them.    ║
╚══════════════════════════════════════════════════════════════════════╝
```

### The 5-Step Flow
```
1. User: "What's the weather?"         → Your app
2. App sends to LLM (with tool schemas) → LLM
3. LLM: {"tool": "get_weather", "args": {"city": "SF"}} → Your app
4. App executes get_weather("SF") → gets "72°F"  → LLM
5. LLM: "It's 72°F in San Francisco!" → User
```

---

## 🔧 Tool Definition Schemas

LLMs need to know what tools exist and how to use them:

```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get current weather for a city",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "The city name, e.g., San Francisco"
        }
      },
      "required": ["city"]
    }
  }
}
```

**Why descriptions matter**: The LLM reads the description to decide which tool to use. A bad description = wrong tool choice!

---

## 🏢 Provider Comparison

| Feature | OpenAI | Anthropic | Google |
|---------|--------|-----------|--------|
| Name | Function Calling | Tool Use | Function Calling |
| Parallel calls | ✅ | ✅ | ✅ |
| Structured output | ✅ via tools | ✅ | ✅ |
| Forced tool use | `tool_choice: required` | `tool_choice: any` | `tool_config` |

LangChain abstracts these differences with `.bind_tools()` — works the same regardless of provider.

---

## 🛡️ Error Handling in Tool Calling

Things that can go wrong:

```
1. Tool not found → LLM invents a tool name
   Fix: Validate tool_call.name against known tools

2. Invalid arguments → LLM passes wrong types
   Fix: Pydantic validation on tool args

3. Tool execution fails → API error, timeout
   Fix: Try/except, return error message, let LLM retry

4. Infinite tool loops → Agent keeps calling tools
   Fix: max_iterations, timeout
```

```python
# Robust tool execution pattern
try:
    result = tool_fn.invoke(tc['args'])
except Exception as e:
    result = f"Error executing {tc['name']}: {str(e)}. Please try a different approach."
```

---

## 🎓 Interview Corner

**Q: How would you build an agent that queries a database and generates reports?**

<details>
<summary>💡 Click for Expert Answer</summary>

1. **Tools**: `query_database(sql)`, `create_chart(data, chart_type)`, `format_report(sections)`
2. **Safety**: SQL injection prevention — use parameterized queries, read-only DB user, query timeout
3. **Flow**: User asks question → Agent writes SQL → Executes → Analyzes results → Generates chart → Formats report
4. **Guardrails**: Whitelist allowed tables, row limit, no DDL statements
5. **Evaluation**: Test with known queries, verify SQL correctness, check report accuracy

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 LLMs REQUEST tool calls; your app EXECUTES them.               │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Tool descriptions are the #1 factor in correct tool selection. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 .bind_tools() works across OpenAI, Anthropic, Google.         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Always handle errors gracefully — return error to LLM.         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Security: validate inputs, use least-privilege, set limits.    │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Hands-On — Agent That Queries a DB and Generates Reports!

---

### 🎉 You Understand Function Calling!
The protocol that connects LLMs to the real world. 🚀